![ab_testing_image](ab_testing_image.jpg)

As a Data Scientist at a leading online travel agency, you’ve been tasked with evaluating the impact of a new search ranking algorithm designed to improve conversion rates. The Product team is considering a full rollout, but only if the experiment shows a clear positive effect on the conversion rate and does not lead to a longer time to book.

They have shared A/B test datasets with session-level booking data (`"sessions_data.csv"`) and user-level control/variant split (`"users_data.csv"`). Your job is to analyze and interpret the results to determine whether the new ranking system delivers a statistically significant improvement and provide a clear, data-driven recommendation.

## `sessions_data.csv`

| column | data type | description | 
|--------|-----------|-------------|
| `session_id` | `string` | Unique session identifier (unique for each row) |
| `user_id` | `string` | Unique user identifier (non logged-in users have missing user_id values; each user can have multiple sessions) |
| `session_start_timestamp` | `string` | When a session started |
| `booking_timestamp` | `string` | When a booking was made (missing if no booking was made during a session) |
| `time_to_booking` | `float` | time from start of the session to booking, in minutes (missing if no booking was made during a session) |
| `conversion` | `integer` | _New column to create:_ did session end up with a booking (0 if booking_timestamp or time_to_booking is Null, otherwise 1) |

<br>

## `users_data.csv`

| column | data type | description | 
|--------|-----------|-------------|
| `user_id` | `string` | Unique user identifier (only logged-in users in this table) |
| `experiment_group` | `string` | control / variant split for the experiment (expected to be equal 50/50) |

<br>

The full on criteria are the following:
- Primary metric (conversion) effect must be statistically significant and show positive effect (increase).
- Guardrail (time_to_booking) effect must either be statistically insignificant or show positive effect (decrease)

In [9]:
import pandas as pd
from scipy.stats import chisquare
from pingouin import ttest
from statsmodels.stats.proportion import proportions_ztest

In [10]:
sessions = pd.read_csv('sessions_data.csv')
users = pd.read_csv('users_data.csv')

In [11]:
sessions.sample(5)

,session_id,user_id,session_start_timestamp,booking_timestamp,time_to_booking
6346,t1Tu9l2kO972Uz1f,syNv9gbME8CQt7dx,2025-01-08 22:25:34.694144011,NaN,NaN
7771,izcCrSqxKQl0xC12,PrNjld3awxW93iM4,2025-01-29 03:16:05.742328405,NaN,NaN
3922,j4jhzaNzckA5fVXE,byI3DLTy4CPUYLLy,2025-01-15 05:20:22.601242781,NaN,NaN
7080,941fqewW8FU9TpmJ,y05btJqpAaHeCjld,2025-01-22 08:59:49.929432869,NaN,NaN
14231,ItayVsPM7ranz42C,W2FfuOzlOv6220lm,2025-01-12 18:56:12.697101116,NaN,NaN


In [12]:
users.sample(5)

,user_id,experiment_group
360,b2NC3hpBBcIBOVzz,control
2494,gGrB63p3s2fIXL4h,variant
5712,fiQKIfxWwIEkMdZ7,variant
1930,caGqs1CeI8vb7hEB,control
6835,DSkP8tlmEZnVX8a7,variant


### Your solution

In [13]:
confidence_level = 0.90  # Set the pre-defined confidence level (90%)
alpha = 1 - confidence_level  # Significance level for hypothesis tests

In [14]:
# by Elok Mutiaraningtyas
# elokmutiaraningtyas@gmail.com

import pandas as pd
from scipy.stats import chisquare
from pingouin import ttest
from statsmodels.stats.proportion import proportions_ztest

sessions = pd.read_csv('sessions_data.csv')
users = pd.read_csv('users_data.csv')

confidence_level = 0.90
alpha = 1 - confidence_level

#1 join datasets and compute conversion
sessions_x_users = sessions.merge(users, on='user_id', how='inner')
sessions_x_users['conversion'] = sessions_x_users['booking_timestamp'].notna().astype(int)

#2 SRM check using chi-square test
group_counts = sessions_x_users['experiment_group'].value_counts()
srm_chi2_pval = chisquare(group_counts.values).pvalue

#3 Statistical significance tests
control = sessions_x_users[sessions_x_users['experiment_group'] == 'control']
variant = sessions_x_users[sessions_x_users['experiment_group'] == 'variant']

# primary metric: conversion (proportions z-test)
count = [variant['conversion'].sum(), control['conversion'].sum()]
nobs = [len(variant), len(control)]
_, pval_primary = proportions_ztest(count, nobs)

# guardrail metric: time_to_booking (t-test on booked sessions only)
control_ttb = control['time_to_booking'].dropna()
variant_ttb = variant['time_to_booking'].dropna()
pval_guardrail = ttest(variant_ttb, control_ttb, correction=True)['p-val'].values[0]

#4 Effect sizes
effect_size_primary = (variant['conversion'].mean() - control['conversion'].mean()) / control['conversion'].mean()
effect_size_guardrail = (variant_ttb.mean() - control_ttb.mean()) / control_ttb.mean()

#5 Decision
primary_significant_positive = (pval_primary < alpha) and (effect_size_primary > 0)
guardrail_ok = (pval_guardrail >= alpha) or (effect_size_guardrail < 0)

decision_full_on = 'yes' if primary_significant_positive and guardrail_ok else 'no'

print(f"SRM p-value: {srm_chi2_pval:.4f}")
print(f"Primary metric p-value: {pval_primary:.4f}, effect size: {effect_size_primary:.4f}")
print(f"Guardrail metric p-value: {pval_guardrail:.4f}, effect size: {effect_size_guardrail:.4f}")
print(f"Decision: {decision_full_on}")

SRM p-value: 0.8524
Primary metric p-value: 0.0002, effect size: 0.1422
Guardrail metric p-value: 0.5365, effect size: -0.0079
Decision: yes
